In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
import nflreadpy as nfl
import polars as pl

player_stats = nfl.load_player_stats([2025])
print(player_stats.shape); print(player_stats.columns)

(19421, 145)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_

In [47]:
players = nfl.load_players()
print(players.shape); print(players.columns)

(25027, 39)
['gsis_id', 'display_name', 'common_first_name', 'first_name', 'last_name', 'short_name', 'football_name', 'suffix', 'esb_id', 'nfl_id', 'pfr_id', 'pff_id', 'otc_id', 'espn_id', 'smart_id', 'birth_date', 'position_group', 'position', 'ngs_position_group', 'ngs_position', 'height', 'weight', 'headshot', 'college_name', 'college_conference', 'jersey_number', 'rookie_season', 'last_season', 'latest_team', 'status', 'ngs_status', 'ngs_status_short_description', 'years_of_experience', 'pff_position', 'pff_status', 'draft_year', 'draft_round', 'draft_pick', 'draft_team']


In [48]:
team_stats = nfl.load_team_stats(seasons=[2025])
print(team_stats.shape); print(team_stats.columns)

(570, 133)
['season', 'week', 'team', 'season_type', 'game_id', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receiving_20', 'receiving_40', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'd

In [49]:
schedules = nfl.load_schedules(seasons=[2025])
print(schedules.shape); print(schedules.columns)

(285, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [50]:
multi_year = nfl.load_player_stats([2023, 2024, 2025])
print(multi_year.group_by("season").len())

shape: (3, 2)
┌────────┬───────┐
│ season ┆ len   │
│ ---    ┆ ---   │
│ i32    ┆ u32   │
╞════════╪═══════╡
│ 2024   ┆ 18981 │
│ 2023   ┆ 18643 │
│ 2025   ┆ 19421 │
└────────┴───────┘


## Season window detection
- Using 2023, 2024,, 2025
- Weighting:  2025 = 50%, 2024 = 30%, 2023 = 20%

In [51]:
import sys
sys.path.append("..")

In [52]:
from src.scoring import load_config, calculate_offensive_points

config = load_config("../league_config.json")
sample = player_stats.filter(
    (pl.col("player_display_name") == "Josh Allen") & (pl.col("week") == 1)
).row(0, named=True)
print(calculate_offensive_points(sample, config))

42.76


In [53]:
print([c for c in player_stats.columns if "int" in c.lower()])

['passing_interceptions', 'def_interceptions', 'def_interception_yards', 'fantasy_points', 'fantasy_points_ppr']


### QB: pass yards/game, pass TD/game, INT/game rush yards/game, rush TD/game, games played
### RB: rush yards/game, rush TD/game, receptions/game, rec yards/game, targets/game, games played
### WR/TE: targets/game, receptions/game, rec yards/game, rec TD/game, games played
### K: FG made/attempted by distance band, XP made
### DST: output of calculate_dst_points(), no separate list needed

In [54]:
draft_picks = nfl.load_draft_picks()
print(draft_picks.columns)
print(draft_picks.head())

['season', 'round', 'pick', 'team', 'gsis_id', 'pfr_player_id', 'cfb_player_id', 'pfr_player_name', 'hof', 'position', 'category', 'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'pass_completions', 'pass_attempts', 'pass_yards', 'pass_tds', 'pass_ints', 'rush_atts', 'rush_yards', 'rush_tds', 'receptions', 'rec_yards', 'rec_tds', 'def_solo_tackles', 'def_ints', 'def_sacks']
shape: (5, 36)
┌────────┬───────┬──────┬──────┬───┬─────────┬──────────────────┬──────────┬───────────┐
│ season ┆ round ┆ pick ┆ team ┆ … ┆ rec_tds ┆ def_solo_tackles ┆ def_ints ┆ def_sacks │
│ ---    ┆ ---   ┆ ---  ┆ ---  ┆   ┆ ---     ┆ ---              ┆ ---      ┆ ---       │
│ i32    ┆ i32   ┆ i32  ┆ str  ┆   ┆ i32     ┆ i32              ┆ i32      ┆ f64       │
╞════════╪═══════╪══════╪══════╪═══╪═════════╪══════════════════╪══════════╪═══════════╡
│ 1980   ┆ 1     ┆ 1    ┆ DET  ┆ … ┆ 5       ┆ null             ┆ null     ┆ null      │
│ 1980   ┆ 1 

In [55]:
rookie_player_stats = nfl.load_player_stats([2021, 2022, 2023, 2024, 2025])

In [56]:
rookie_stats = rookie_player_stats.join(
    draft_picks.select(["gsis_id", "season", "round"]),
    left_on="player_id", right_on="gsis_id"
).filter(pl.col("season") == pl.col("season_right"))

In [57]:
rookie_stats.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'game_id',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_

In [58]:
from src.scoring import load_config, calculate_offensive_points
config = load_config("../league_config.json")

offense_positions = ["QB", "RB", "WR", "TE"]
rookie_stats = rookie_stats.filter(pl.col("position").is_in(offense_positions))

rookie_stats = rookie_stats.with_columns(
    pl.struct(rookie_stats.columns)
    .map_elements(lambda row: calculate_offensive_points(row, config))
    .alias("fantasy_points")
)

In [59]:
summary = (
    rookie_stats.group_by(["position", "round"])
    .agg([
        pl.col("fantasy_points").mean().alias("avg_points_per_game"),
        pl.col("player_id").n_unique().alias("num_players"),
    ])
    .sort(["position", "round"])
)
print(summary)

shape: (28, 4)
┌──────────┬───────┬─────────────────────┬─────────────┐
│ position ┆ round ┆ avg_points_per_game ┆ num_players │
│ ---      ┆ ---   ┆ ---                 ┆ ---         │
│ str      ┆ i32   ┆ f64                 ┆ u32         │
╞══════════╪═══════╪═════════════════════╪═════════════╡
│ QB       ┆ 1     ┆ 16.345545           ┆ 16          │
│ QB       ┆ 2     ┆ 14.799              ┆ 2           │
│ QB       ┆ 3     ┆ 8.781579            ┆ 6           │
│ QB       ┆ 4     ┆ 12.59125            ┆ 3           │
│ QB       ┆ 5     ┆ 6.747647            ┆ 7           │
│ QB       ┆ 6     ┆ 6.348889            ┆ 3           │
│ QB       ┆ 7     ┆ 10.548333           ┆ 3           │
│ RB       ┆ 1     ┆ 15.2925             ┆ 5           │
│ RB       ┆ 2     ┆ 10.584733           ┆ 9           │
│ RB       ┆ 3     ┆ 6.313178            ┆ 12          │
│ RB       ┆ 4     ┆ 6.183607            ┆ 23          │
│ RB       ┆ 5     ┆ 4.377647            ┆ 20          │
│ RB       ┆ 6  

In [60]:
pl.Config.set_tbl_rows(-1)
print(summary)

shape: (28, 4)
┌──────────┬───────┬─────────────────────┬─────────────┐
│ position ┆ round ┆ avg_points_per_game ┆ num_players │
│ ---      ┆ ---   ┆ ---                 ┆ ---         │
│ str      ┆ i32   ┆ f64                 ┆ u32         │
╞══════════╪═══════╪═════════════════════╪═════════════╡
│ QB       ┆ 1     ┆ 16.345545           ┆ 16          │
│ QB       ┆ 2     ┆ 14.799              ┆ 2           │
│ QB       ┆ 3     ┆ 8.781579            ┆ 6           │
│ QB       ┆ 4     ┆ 12.59125            ┆ 3           │
│ QB       ┆ 5     ┆ 6.747647            ┆ 7           │
│ QB       ┆ 6     ┆ 6.348889            ┆ 3           │
│ QB       ┆ 7     ┆ 10.548333           ┆ 3           │
│ RB       ┆ 1     ┆ 15.2925             ┆ 5           │
│ RB       ┆ 2     ┆ 10.584733           ┆ 9           │
│ RB       ┆ 3     ┆ 6.313178            ┆ 12          │
│ RB       ┆ 4     ┆ 6.183607            ┆ 23          │
│ RB       ┆ 5     ┆ 4.377647            ┆ 20          │
│ RB       ┆ 6  

In [61]:
dupes = players.group_by("gsis_id").len().filter(pl.col("len") > 1)
print(dupes)

mid_season_team_changes = (
    player_stats.group_by(["player_id", "season"])
    .agg(pl.col("team").n_unique().alias("n_teams"))
    .filter(pl.col("n_teams") > 1)
)
print(mid_season_team_changes)

shape: (0, 2)
┌─────────┬─────┐
│ gsis_id ┆ len │
│ ---     ┆ --- │
│ str     ┆ u32 │
╞═════════╪═════╡
└─────────┴─────┘
shape: (89, 3)
┌────────────┬────────┬─────────┐
│ player_id  ┆ season ┆ n_teams │
│ ---        ┆ ---    ┆ ---     │
│ str        ┆ i32    ┆ u32     │
╞════════════╪════════╪═════════╡
│ 00-0035697 ┆ 2025   ┆ 2       │
│ 00-0037242 ┆ 2025   ┆ 2       │
│ 00-0035406 ┆ 2025   ┆ 2       │
│ 00-0038496 ┆ 2025   ┆ 2       │
│ 00-0037281 ┆ 2025   ┆ 2       │
│ 00-0040527 ┆ 2025   ┆ 2       │
│ 00-0036980 ┆ 2025   ┆ 2       │
│ 00-0035253 ┆ 2025   ┆ 2       │
│ 00-0035895 ┆ 2025   ┆ 2       │
│ 00-0036294 ┆ 2025   ┆ 2       │
│ 00-0036904 ┆ 2025   ┆ 2       │
│ 00-0039235 ┆ 2025   ┆ 2       │
│ 00-0033702 ┆ 2025   ┆ 2       │
│ 00-0026158 ┆ 2025   ┆ 2       │
│ 00-0037340 ┆ 2025   ┆ 2       │
│ 00-0034968 ┆ 2025   ┆ 3       │
│ 00-0035633 ┆ 2025   ┆ 2       │
│ 00-0037614 ┆ 2025   ┆ 2       │
│ 00-0036380 ┆ 2025   ┆ 2       │
│ 00-0036305 ┆ 2025   ┆ 2       │
│ 00-0037830 

In [62]:
print(
    player_stats.filter(pl.col("player_id").is_null())
    .select(["player_name", "position", "team", "week"])
    .head(20)
)

shape: (20, 4)
┌─────────────┬──────────┬──────┬──────┐
│ player_name ┆ position ┆ team ┆ week │
│ ---         ┆ ---      ┆ ---  ┆ ---  │
│ str         ┆ str      ┆ str  ┆ i32  │
╞═════════════╪══════════╪══════╪══════╡
│ null        ┆ null     ┆ DAL  ┆ 1    │
│ null        ┆ null     ┆ GB   ┆ 2    │
│ null        ┆ null     ┆ MIA  ┆ 3    │
│ null        ┆ null     ┆ ARI  ┆ 4    │
│ null        ┆ null     ┆ SF   ┆ 5    │
│ null        ┆ null     ┆ PHI  ┆ 6    │
│ null        ┆ null     ┆ PIT  ┆ 7    │
│ null        ┆ null     ┆ MIN  ┆ 8    │
│ null        ┆ null     ┆ MIA  ┆ 9    │
│ null        ┆ null     ┆ LV   ┆ 10   │
│ null        ┆ null     ┆ NYJ  ┆ 11   │
│ null        ┆ null     ┆ HOU  ┆ 12   │
│ null        ┆ null     ┆ GB   ┆ 13   │
│ null        ┆ null     ┆ DET  ┆ 14   │
│ null        ┆ null     ┆ TB   ┆ 15   │
│ null        ┆ null     ┆ LA   ┆ 16   │
│ null        ┆ null     ┆ DAL  ┆ 17   │
│ null        ┆ null     ┆ TB   ┆ 18   │
│ null        ┆ null     ┆ CAR  ┆ 19   │
│

In [63]:
from src.features import load_veteran_stats, aggregate_season_stats, apply_season_weighting

raw = load_veteran_stats([2023, 2024, 2025])
season_stats = aggregate_season_stats(raw)
weighted = apply_season_weighting(season_stats)

In [64]:
print(weighted.schema)

Schema({'passing_yards_per_game': Float64, 'passing_tds_per_game': Float64, 'passing_interceptions_per_game': Float64, 'rushing_yards_per_game': Float64, 'rushing_tds_per_game': Float64, 'receptions_per_game': Float64, 'targets_per_game': Float64, 'receiving_yards_per_game': Float64, 'receiving_tds_per_game': Float64, 'fantasy_points_per_game': Float64, 'player_id': String, 'player_name': String, 'position': String, 'games_played': Int64})


In [65]:
import nflreadpy as nfl
print([f for f in dir(nfl) if not f.startswith("_")])

['cache', 'clear_cache', 'config', 'downloader', 'get_current_season', 'get_current_week', 'load_combine', 'load_contracts', 'load_depth_charts', 'load_draft_picks', 'load_ff_opportunity', 'load_ff_playerids', 'load_ff_rankings', 'load_ffverse', 'load_ftn_charting', 'load_injuries', 'load_nextgen_stats', 'load_officials', 'load_participation', 'load_pbp', 'load_pfr_advstats', 'load_player_stats', 'load_players', 'load_rosters', 'load_rosters_weekly', 'load_schedules', 'load_snap_counts', 'load_stats', 'load_team_stats', 'load_teams', 'load_trades', 'utils_date', 'version']


In [66]:
import polars as pl
import nflreadpy as nfl

# Match this range to whatever seasons your playcaller_history.csv actually covers
seasons = list(range(2020, 2026))

team_stats = nfl.load_team_stats(seasons=seasons)
tendency = (
    team_stats.filter(pl.col("season_type") == "REG")
    .group_by(["team", "season"])
    .agg([
        pl.col("attempts").sum().alias("pass_att_season"),
        pl.col("carries").sum().alias("rush_att_season"),
        pl.col("week").n_unique().alias("games"),
    ])
    .with_columns([
        (pl.col("pass_att_season") / pl.col("games")).alias("pass_att_pg"),
        (pl.col("rush_att_season") / pl.col("games")).alias("rush_att_pg"),
    ])
)

player_stats = nfl.load_player_stats(seasons)
primary_qb = (
    player_stats.filter((pl.col("position") == "QB") & (pl.col("season_type") == "REG"))
    .group_by(["team", "season", "player_id"])
    .agg(pl.col("attempts").sum().alias("attempts"))
    .sort("attempts", descending=True)
    .group_by(["team", "season"])
    .first()
    .select(["team", "season", "player_id"])
)

prior_qb = primary_qb.with_columns((pl.col("season") + 1).alias("season")).rename({"player_id": "prior_qb_id"})
qb_change = primary_qb.join(prior_qb, on=["team", "season"], how="left").with_columns(
    (pl.col("player_id") != pl.col("prior_qb_id")).fill_null(True).alias("qb_changed")
)

playcallers = pl.read_csv("../playcaller_history.csv")
coach_change = playcallers.select(["team", "season", "changed_from_prior_year"]).with_columns(
    pl.col("changed_from_prior_year").cast(pl.String).str.to_lowercase().eq("true").alias("changed_from_prior_year")
).rename({"changed_from_prior_year": "coach_changed"})
prior_tendency = tendency.with_columns((pl.col("season") + 1).alias("season")).select(
    ["team", "season", "pass_att_pg", "rush_att_pg"]
).rename({"pass_att_pg": "prior_pass_att_pg", "rush_att_pg": "prior_rush_att_pg"})

combined = (
    tendency.join(qb_change.select(["team", "season", "qb_changed"]), on=["team", "season"], how="left")
    .join(coach_change, on=["team", "season"], how="left")
    .join(prior_tendency, on=["team", "season"], how="left")
    .with_columns([
        (pl.col("pass_att_pg") - pl.col("prior_pass_att_pg")).abs().alias("pass_shift"),
        (pl.col("rush_att_pg") - pl.col("prior_rush_att_pg")).abs().alias("rush_shift"),
        (pl.col("qb_changed").fill_null(False).cast(pl.Int8) + pl.col("coach_changed").fill_null(False).cast(pl.Int8)).alias("continuity_score"),
    ])
)

result = (
    combined.group_by("continuity_score")
    .agg([
        pl.col("pass_shift").mean().alias("avg_pass_shift"),
        pl.col("rush_shift").mean().alias("avg_rush_shift"),
        pl.len().alias("n"),
    ])
    .sort("continuity_score")
)
print(result)

shape: (3, 4)
┌──────────────────┬────────────────┬────────────────┬─────┐
│ continuity_score ┆ avg_pass_shift ┆ avg_rush_shift ┆ n   │
│ ---              ┆ ---            ┆ ---            ┆ --- │
│ i8               ┆ f64            ┆ f64            ┆ u32 │
╞══════════════════╪════════════════╪════════════════╪═════╡
│ 0                ┆ 2.756819       ┆ 1.611361       ┆ 62  │
│ 1                ┆ 3.075765       ┆ 2.773698       ┆ 106 │
│ 2                ┆ 3.655331       ┆ 3.127604       ┆ 24  │
└──────────────────┴────────────────┴────────────────┴─────┘


In [67]:
snap_counts = nfl.load_snap_counts(seasons=[2025])
print(snap_counts.shape)
print(snap_counts.columns)
print(snap_counts.head(10))

(26612, 16)
['game_id', 'pfr_game_id', 'season', 'game_type', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']
shape: (10, 16)
┌─────────────┬─────────────┬────────┬───────────┬───┬────────────┬────────────┬──────────┬────────┐
│ game_id     ┆ pfr_game_id ┆ season ┆ game_type ┆ … ┆ defense_sn ┆ defense_pc ┆ st_snaps ┆ st_pct │
│ ---         ┆ ---         ┆ ---    ┆ ---       ┆   ┆ aps        ┆ t          ┆ ---      ┆ ---    │
│ str         ┆ str         ┆ i32    ┆ str       ┆   ┆ ---        ┆ ---        ┆ f64      ┆ f64    │
│             ┆             ┆        ┆           ┆   ┆ f64        ┆ f64        ┆          ┆        │
╞═════════════╪═════════════╪════════╪═══════════╪═══╪════════════╪════════════╪══════════╪════════╡
│ 2025_01_ARI ┆ 202509070no ┆ 2025   ┆ REG       ┆ … ┆ 0.0        ┆ 0.0        ┆ 5.0      ┆ 0.19   │
│ _NO         ┆ r           ┆        ┆           ┆   ┆        

In [68]:
print(snap_counts["position"].unique().sort())

shape: (22,)
Series: 'position' [str]
[
	"C"
	"CB"
	"DB"
	"DE"
	"DL"
	"DT"
	"FB"
	"FS"
	"G"
	"HB"
	"K"
	"LB"
	"LS"
	"NT"
	"OL"
	"P"
	"QB"
	"RB"
	"S"
	"T"
	"TE"
	"WR"
]


In [69]:
playerids = nfl.load_ff_playerids()
print(playerids.shape)
print(playerids.columns)
print(playerids.head(5))

(12468, 35)
['mfl_id', 'sportradar_id', 'fantasypros_id', 'gsis_id', 'pff_id', 'sleeper_id', 'nfl_id', 'espn_id', 'yahoo_id', 'fleaflicker_id', 'cbs_id', 'pfr_id', 'cfbref_id', 'rotowire_id', 'rotoworld_id', 'ktc_id', 'stats_id', 'stats_global_id', 'fantasy_data_id', 'swish_id', 'name', 'merge_name', 'position', 'team', 'birthdate', 'age', 'draft_year', 'draft_round', 'draft_pick', 'draft_ovr', 'twitter_username', 'height', 'weight', 'college', 'db_season']
shape: (5, 35)
┌────────┬──────────────┬──────────────┬───────────┬───┬────────┬────────┬─────────────┬───────────┐
│ mfl_id ┆ sportradar_i ┆ fantasypros_ ┆ gsis_id   ┆ … ┆ height ┆ weight ┆ college     ┆ db_season │
│ ---    ┆ d            ┆ id           ┆ ---       ┆   ┆ ---    ┆ ---    ┆ ---         ┆ ---       │
│ i64    ┆ ---          ┆ ---          ┆ str       ┆   ┆ i64    ┆ i64    ┆ str         ┆ i64       │
│        ┆ str          ┆ str          ┆           ┆   ┆        ┆        ┆             ┆           │
╞════════╪════════

In [70]:
depth_charts = nfl.load_depth_charts(seasons=[2026])
print(depth_charts.shape)
print(depth_charts.columns)
print(depth_charts.filter(pl.col("pos_abb") == "QB").head(15))
print(depth_charts.filter(pl.col("pos_abb") == "QB").select(["team", "player_name", "gsis_id", "pos_rank", "dt"]).sort(["team", "pos_rank"]).head(20))

(384820, 12)
['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']
shape: (15, 12)
┌───────────────┬──────┬───────────────┬─────────┬───┬─────────────┬─────────┬──────────┬──────────┐
│ dt            ┆ team ┆ player_name   ┆ espn_id ┆ … ┆ pos_name    ┆ pos_abb ┆ pos_slot ┆ pos_rank │
│ ---           ┆ ---  ┆ ---           ┆ ---     ┆   ┆ ---         ┆ ---     ┆ ---      ┆ ---      │
│ str           ┆ str  ┆ str           ┆ str     ┆   ┆ str         ┆ str     ┆ i32      ┆ i32      │
╞═══════════════╪══════╪═══════════════╪═════════╪═══╪═════════════╪═════════╪══════════╪══════════╡
│ 2026-07-30T09 ┆ ARI  ┆ Jacoby        ┆ 2578570 ┆ … ┆ Quarterback ┆ QB      ┆ 9        ┆ 1        │
│ :27:38Z       ┆      ┆ Brissett      ┆         ┆   ┆             ┆         ┆          ┆          │
│ 2026-07-30T09 ┆ ARI  ┆ Gardner       ┆ 4038524 ┆ … ┆ Quarterback ┆ QB      ┆ 9        ┆ 2        │
│ :27:38Z       ┆      ┆ Mins

In [71]:
teams = nfl.load_teams()
print(teams.shape)
print(teams.columns)
print(teams.head())

team_stats_codes = set(nfl.load_team_stats(seasons=[2025])["team"].unique().to_list())
player_codes = set(nfl.load_players()["latest_team"].drop_nulls().unique().to_list())

print("In load_players() but not in team_stats:", player_codes - team_stats_codes)
print("In team_stats but not in load_players():", team_stats_codes - player_codes)

(36, 16)
['team_abbr', 'team_name', 'team_id', 'team_nick', 'team_conf', 'team_division', 'team_color', 'team_color2', 'team_color3', 'team_color4', 'team_logo_wikipedia', 'team_logo_espn', 'team_wordmark', 'team_conference_logo', 'team_league_logo', 'team_logo_squared']
shape: (5, 16)
┌───────────┬────────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ team_abbr ┆ team_name  ┆ team_id ┆ team_nick ┆ … ┆ team_word ┆ team_conf ┆ team_leag ┆ team_logo │
│ ---       ┆ ---        ┆ ---     ┆ ---       ┆   ┆ mark      ┆ erence_lo ┆ ue_logo   ┆ _squared  │
│ str       ┆ str        ┆ str     ┆ str       ┆   ┆ ---       ┆ go        ┆ ---       ┆ ---       │
│           ┆            ┆         ┆           ┆   ┆ str       ┆ ---       ┆ str       ┆ str       │
│           ┆            ┆         ┆           ┆   ┆           ┆ str       ┆           ┆           │
╞═══════════╪════════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ ARI 

In [72]:
from src.rookies import get_current_rookie_class
rc = get_current_rookie_class()
print(rc.filter(pl.col("player_name") == "Carson Beck"))

shape: (1, 5)
┌────────────┬─────────────┬──────────┬──────┬───────┐
│ player_id  ┆ player_name ┆ position ┆ team ┆ round │
│ ---        ┆ ---         ┆ ---      ┆ ---  ┆ ---   │
│ str        ┆ str         ┆ str      ┆ str  ┆ i32   │
╞════════════╪═════════════╪══════════╪══════╪═══════╡
│ 00-0041561 ┆ Carson Beck ┆ QB       ┆ ARI  ┆ 3     │
└────────────┴─────────────┴──────────┴──────┴───────┘


In [73]:
import polars as pl
check = pl.read_csv("../data/player_features.csv")
print(check.filter(pl.col("player_name") == "Carson Beck"))

shape: (1, 37)
┌─────────────┬─────────────┬─────────────┬─────────────┬───┬─────────┬───────────┬──────┬─────────┐
│ passing_yar ┆ passing_tds ┆ passing_int ┆ rushing_yar ┆ … ┆ adp_low ┆ adp_stdev ┆ bye  ┆ has_adp │
│ ds_per_game ┆ _per_game   ┆ erceptions_ ┆ ds_per_game ┆   ┆ ---     ┆ ---       ┆ ---  ┆ ---     │
│ ---         ┆ ---         ┆ per_game    ┆ ---         ┆   ┆ i64     ┆ f64       ┆ i64  ┆ bool    │
│ f64         ┆ f64         ┆ ---         ┆ f64         ┆   ┆         ┆           ┆      ┆         │
│             ┆             ┆ f64         ┆             ┆   ┆         ┆           ┆      ┆         │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══╪═════════╪═══════════╪══════╪═════════╡
│ null        ┆ null        ┆ null        ┆ null        ┆ … ┆ null    ┆ null      ┆ null ┆ false   │
└─────────────┴─────────────┴─────────────┴─────────────┴───┴─────────┴───────────┴──────┴─────────┘


In [74]:
with pl.Config(set_tbl_cols=-1):
    print(check.filter(pl.col("player_name") == "Carson Beck"))

shape: (1, 37)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pas ┆ pas ┆ pas ┆ rus ┆ rus ┆ rec ┆ tar ┆ rec ┆ rec ┆ fan ┆ pla ┆ pla ┆ pos ┆ gam ┆ tea ┆ is_ ┆ bas ┆ rou ┆ rou ┆ pos ┆ is_ ┆ pas ┆ rus ┆ qb_ ┆ coa ┆ con ┆ ret ┆ pos ┆ rec ┆ adp ┆ adp ┆ tim ┆ adp ┆ adp ┆ adp ┆ bye ┆ has │
│ sin ┆ sin ┆ sin ┆ hin ┆ hin ┆ ept ┆ get ┆ eiv ┆ eiv ┆ tas ┆ yer ┆ yer ┆ iti ┆ es_ ┆ m   ┆ roo ┆ eli ┆ nd  ┆ nd_ ┆ _ra ┆ sta ┆ s_a ┆ h_a ┆ cha ┆ ch_ ┆ tin ┆ urn ┆ iti ┆ ent ┆ --- ┆ _fo ┆ es_ ┆ _hi ┆ _lo ┆ _st ┆ --- ┆ _ad │
│ g_y ┆ g_t ┆ g_i ┆ g_y ┆ g_t ┆ ion ┆ s_p ┆ ing ┆ ing ┆ y_p ┆ _id ┆ _na ┆ on  ┆ pla ┆ --- ┆ kie ┆ ne_ ┆ --- ┆ buc ┆ nk  ┆ rte ┆ tt_ ┆ tt_ ┆ nge ┆ cha ┆ uit ┆ ing ┆ on_ ┆ _ma ┆ f64 ┆ rma ┆ dra ┆ gh  ┆ w   ┆ dev ┆ i64 ┆ p   │
│ ard ┆ ds_ ┆ nte ┆ ard ┆ ds_ ┆ s_p ┆ er_ ┆ _ya ┆ _td ┆ oin ┆ --- ┆ me  ┆ --- ┆ yed ┆ str

In [75]:
beck_player = nfl.load_players().filter(pl.col("display_name").str.contains("Beck"))
print(beck_player.select(["gsis_id", "display_name", "draft_year", "draft_round", "draft_pick", "latest_team", "rookie_season"]))

shape: (23, 7)
┌────────────┬───────────────┬────────────┬─────────────┬────────────┬─────────────┬───────────────┐
│ gsis_id    ┆ display_name  ┆ draft_year ┆ draft_round ┆ draft_pick ┆ latest_team ┆ rookie_season │
│ ---        ┆ ---           ┆ ---        ┆ ---         ┆ ---        ┆ ---         ┆ ---           │
│ str        ┆ str           ┆ i32        ┆ i32         ┆ i32        ┆ str         ┆ i32           │
╞════════════╪═══════════════╪════════════╪═════════════╪════════════╪═════════════╪═══════════════╡
│ 00-0034959 ┆ Andrew Beck   ┆ null       ┆ null        ┆ null       ┆ NYJ         ┆ 2019          │
│ 00-0041561 ┆ Carson Beck   ┆ null       ┆ null        ┆ null       ┆ AZ          ┆ 2026          │
│ 00-0025427 ┆ John Beck     ┆ 2007       ┆ 2           ┆ 40         ┆ HOU         ┆ 2007          │
│ 00-0023525 ┆ Jordan Beck   ┆ 2005       ┆ 3           ┆ 90         ┆ DEN         ┆ 2005          │
│ 00-0019731 ┆ Matt Beck     ┆ null       ┆ null        ┆ null       ┆ LA   

In [76]:
draft_picks_2026 = nfl.load_draft_picks().filter(pl.col("season") == 2026)
print("load_draft_picks() rows for 2026:", draft_picks_2026.shape[0])

players_2026 = nfl.load_players().filter(pl.col("rookie_season") == 2026)
print("Total 2026 rookies in load_players():", players_2026.shape[0])
print("...with a null draft_round:", players_2026.filter(pl.col("draft_round").is_null()).shape[0])
print("...with a real draft_round:", players_2026.filter(pl.col("draft_round").is_not_null()).shape[0])

load_draft_picks() rows for 2026: 257
Total 2026 rookies in load_players(): 671
...with a null draft_round: 671
...with a real draft_round: 0


In [77]:
players_2026 = nfl.load_players().filter(pl.col("rookie_season") == 2026).select(["gsis_id", "pfr_id", "display_name"])
draft_2026 = nfl.load_draft_picks().filter(pl.col("season") == 2026).select(["gsis_id", "pfr_player_id", "round"])

matched_on_gsis = players_2026.join(draft_2026, on="gsis_id", how="inner")
print("Matched via gsis_id:", matched_on_gsis.shape[0])

matched_on_pfr = players_2026.join(draft_2026, left_on="pfr_id", right_on="pfr_player_id", how="inner")
print("Matched via pfr_id:", matched_on_pfr.shape[0])

Matched via gsis_id: 8
Matched via pfr_id: 230


In [78]:
import polars as pl
check = pl.read_csv("../data/player_features.csv")
with pl.Config(set_tbl_cols=-1):
    print(check.filter(pl.col("player_name") == "Carson Beck"))

shape: (1, 37)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pas ┆ pas ┆ pas ┆ rus ┆ rus ┆ rec ┆ tar ┆ rec ┆ rec ┆ fan ┆ pla ┆ pla ┆ pos ┆ gam ┆ tea ┆ is_ ┆ bas ┆ rou ┆ rou ┆ pos ┆ is_ ┆ pas ┆ rus ┆ qb_ ┆ coa ┆ con ┆ ret ┆ pos ┆ rec ┆ adp ┆ adp ┆ tim ┆ adp ┆ adp ┆ adp ┆ bye ┆ has │
│ sin ┆ sin ┆ sin ┆ hin ┆ hin ┆ ept ┆ get ┆ eiv ┆ eiv ┆ tas ┆ yer ┆ yer ┆ iti ┆ es_ ┆ m   ┆ roo ┆ eli ┆ nd  ┆ nd_ ┆ _ra ┆ sta ┆ s_a ┆ h_a ┆ cha ┆ ch_ ┆ tin ┆ urn ┆ iti ┆ ent ┆ --- ┆ _fo ┆ es_ ┆ _hi ┆ _lo ┆ _st ┆ --- ┆ _ad │
│ g_y ┆ g_t ┆ g_i ┆ g_y ┆ g_t ┆ ion ┆ s_p ┆ ing ┆ ing ┆ y_p ┆ _id ┆ _na ┆ on  ┆ pla ┆ --- ┆ kie ┆ ne_ ┆ --- ┆ buc ┆ nk  ┆ rte ┆ tt_ ┆ tt_ ┆ nge ┆ cha ┆ uit ┆ ing ┆ on_ ┆ _ma ┆ f64 ┆ rma ┆ dra ┆ gh  ┆ w   ┆ dev ┆ i64 ┆ p   │
│ ard ┆ ds_ ┆ nte ┆ ard ┆ ds_ ┆ s_p ┆ er_ ┆ _ya ┆ _td ┆ oin ┆ --- ┆ me  ┆ --- ┆ yed ┆ str

In [79]:
players_2026 = nfl.load_players().filter(pl.col("rookie_season") == 2026).select(["gsis_id", "pfr_id", "display_name"])
draft_2026 = nfl.load_draft_picks().filter(pl.col("season") == 2026).select(["gsis_id", "pfr_player_id", "round"])

beck_check = players_2026.filter(pl.col("display_name").str.contains("Beck")).join(
    draft_2026, left_on="pfr_id", right_on="pfr_player_id", how="left"
)
print(beck_check)

shape: (1, 5)
┌────────────┬────────┬──────────────┬───────────────┬───────┐
│ gsis_id    ┆ pfr_id ┆ display_name ┆ gsis_id_right ┆ round │
│ ---        ┆ ---    ┆ ---          ┆ ---           ┆ ---   │
│ str        ┆ str    ┆ str          ┆ str           ┆ i32   │
╞════════════╪════════╪══════════════╪═══════════════╪═══════╡
│ 00-0041561 ┆ null   ┆ Carson Beck  ┆ null          ┆ null  │
└────────────┴────────┴──────────────┴───────────────┴───────┘


In [80]:
players_2026 = nfl.load_players().filter(
    (pl.col("rookie_season") == 2026) & (pl.col("position").is_in(["QB", "RB", "WR", "TE"]))
).select(["gsis_id", "pfr_id", "display_name", "position"])

draft_2026 = nfl.load_draft_picks().filter(pl.col("season") == 2026).select(
    ["pfr_player_name", "pfr_player_id", "round", "pick"]
)

unmatched = players_2026.filter(pl.col("pfr_id").is_null()) 
print("Offense rookies with no pfr_id at all:", unmatched.shape[0])
print(unmatched.select(["display_name", "position"]).sort("position"))

matched_offense = players_2026.filter(pl.col("pfr_id").is_not_null()).join(
    draft_2026, left_on="pfr_id", right_on="pfr_player_id", how="left"
)
still_unmatched = matched_offense.filter(pl.col("round").is_null())
print("Have a pfr_id but still didn't match a draft pick:", still_unmatched.shape[0])
print(still_unmatched.select(["display_name", "position"]).sort("position"))

Offense rookies with no pfr_id at all: 99
shape: (99, 2)
┌─────────────────────────┬──────────┐
│ display_name            ┆ position │
│ ---                     ┆ ---      │
│ str                     ┆ str      │
╞═════════════════════════╪══════════╡
│ Joey Aguilar            ┆ QB       │
│ Carson Beck             ┆ QB       │
│ Matthew Caldwell        ┆ QB       │
│ Jacob Clark             ┆ QB       │
│ Haynes King             ┆ QB       │
│ Jack Strand             ┆ QB       │
│ Jackson Acker           ┆ RB       │
│ Damon Bankston          ┆ RB       │
│ Coleman Bennett         ┆ RB       │
│ Kadarius Calloway       ┆ RB       │
│ Anderson Castle         ┆ RB       │
│ Miles Davis             ┆ RB       │
│ Gregory Desrosiers      ┆ RB       │
│ TJ Harden               ┆ RB       │
│ DJ Herman               ┆ RB       │
│ Dontae McMillan         ┆ RB       │
│ Myles Montgomery        ┆ RB       │
│ Jaden Nixon             ┆ RB       │
│ Jaydn Ott               ┆ RB       │
│ Kejon

In [81]:
draft_2026_named = nfl.load_draft_picks().filter(pl.col("season") == 2026).select(
    ["pfr_player_name", "round", "position"]
).rename({"pfr_player_name": "display_name"})

unmatched_names = pl.concat([
    unmatched.select(["display_name", "position"]),
    still_unmatched.select(["display_name", "position"]),
])

name_matched = unmatched_names.join(draft_2026_named, on=["display_name", "position"], how="inner")
print("Recovered via exact name match (these were actually drafted):", name_matched.shape[0])
print(name_matched.sort("round"))

still_missing = unmatched_names.join(draft_2026_named, on=["display_name", "position"], how="anti")
print("Still unmatched (likely genuinely undrafted, or a name-spelling mismatch):", still_missing.shape[0])

Recovered via exact name match (these were actually drafted): 7
shape: (7, 3)
┌────────────────────┬──────────┬───────┐
│ display_name       ┆ position ┆ round │
│ ---                ┆ ---      ┆ ---   │
│ str                ┆ str      ┆ i32   │
╞════════════════════╪══════════╪═══════╡
│ De'Zhaun Stribling ┆ WR       ┆ 2     │
│ Carson Beck        ┆ QB       ┆ 3     │
│ Oscar Delp         ┆ TE       ┆ 3     │
│ Colbie Young       ┆ WR       ┆ 4     │
│ Nicholas Singleton ┆ RB       ┆ 5     │
│ Joe Royer          ┆ TE       ┆ 5     │
│ Deion Burks        ┆ WR       ┆ 7     │
└────────────────────┴──────────┴───────┘
Still unmatched (likely genuinely undrafted, or a name-spelling mismatch): 144


In [82]:
import polars as pl
check = pl.read_csv("../data/player_features.csv")
with pl.Config(set_tbl_cols=-1):
    print(check.filter(pl.col("player_name") == "Carson Beck"))

shape: (1, 37)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pas ┆ pas ┆ pas ┆ rus ┆ rus ┆ rec ┆ tar ┆ rec ┆ rec ┆ fan ┆ pla ┆ pla ┆ pos ┆ gam ┆ tea ┆ is_ ┆ bas ┆ rou ┆ rou ┆ pos ┆ is_ ┆ pas ┆ rus ┆ qb_ ┆ coa ┆ con ┆ ret ┆ pos ┆ rec ┆ adp ┆ adp ┆ tim ┆ adp ┆ adp ┆ adp ┆ bye ┆ has │
│ sin ┆ sin ┆ sin ┆ hin ┆ hin ┆ ept ┆ get ┆ eiv ┆ eiv ┆ tas ┆ yer ┆ yer ┆ iti ┆ es_ ┆ m   ┆ roo ┆ eli ┆ nd  ┆ nd_ ┆ _ra ┆ sta ┆ s_a ┆ h_a ┆ cha ┆ ch_ ┆ tin ┆ urn ┆ iti ┆ ent ┆ --- ┆ _fo ┆ es_ ┆ _hi ┆ _lo ┆ _st ┆ --- ┆ _ad │
│ g_y ┆ g_t ┆ g_i ┆ g_y ┆ g_t ┆ ion ┆ s_p ┆ ing ┆ ing ┆ y_p ┆ _id ┆ _na ┆ on  ┆ pla ┆ --- ┆ kie ┆ ne_ ┆ --- ┆ buc ┆ nk  ┆ rte ┆ tt_ ┆ tt_ ┆ nge ┆ cha ┆ uit ┆ ing ┆ on_ ┆ _ma ┆ f64 ┆ rma ┆ dra ┆ gh  ┆ w   ┆ dev ┆ i64 ┆ p   │
│ ard ┆ ds_ ┆ nte ┆ ard ┆ ds_ ┆ s_p ┆ er_ ┆ _ya ┆ _td ┆ oin ┆ --- ┆ me  ┆ --- ┆ yed ┆ str

In [83]:
injuries = nfl.load_injuries(seasons=[2025])
print(injuries.shape)
print(injuries.columns)
print(injuries.filter(pl.col("full_name").str.contains("Mahomes")))

(6068, 16)
['season', 'season_type', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status']
shape: (7, 16)
┌────────┬─────────────┬───────────┬──────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ season ┆ season_type ┆ game_type ┆ team ┆ … ┆ report_stat ┆ practice_p ┆ practice_s ┆ practice_s │
│ ---    ┆ ---         ┆ ---       ┆ ---  ┆   ┆ us          ┆ rimary_inj ┆ econdary_i ┆ tatus      │
│ i32    ┆ str         ┆ str       ┆ str  ┆   ┆ ---         ┆ ury        ┆ njury      ┆ ---        │
│        ┆             ┆           ┆      ┆   ┆ str         ┆ ---        ┆ ---        ┆ str        │
│        ┆             ┆           ┆      ┆   ┆             ┆ str        ┆ str        ┆            │
╞════════╪═════════════╪═══════════╪══════╪═══╪═════════════╪════════════╪════════════╪════════════╡
│ 202

In [84]:
with pl.Config(set_tbl_cols=-1):
    print(
        injuries.filter(pl.col("full_name").str.contains("Mahomes"))
        .select(["week", "report_primary_injury", "report_secondary_injury", "report_status", "practice_status"])
        .sort("week")
    )

shape: (7, 5)
┌──────┬───────────────────────┬─────────────────────────┬───────────────┬─────────────────────────┐
│ week ┆ report_primary_injury ┆ report_secondary_injury ┆ report_status ┆ practice_status         │
│ ---  ┆ ---                   ┆ ---                     ┆ ---           ┆ ---                     │
│ i32  ┆ str                   ┆ str                     ┆ str           ┆ str                     │
╞══════╪═══════════════════════╪═════════════════════════╪═══════════════╪═════════════════════════╡
│ 3    ┆ null                  ┆ null                    ┆ null          ┆ Full Participation in   │
│      ┆                       ┆                         ┆               ┆ Practice                │
│ 4    ┆ null                  ┆ null                    ┆ null          ┆ Full Participation in   │
│      ┆                       ┆                         ┆               ┆ Practice                │
│ 5    ┆ null                  ┆ null                    ┆ null          ┆ Fu

In [85]:
rosters_weekly = nfl.load_rosters_weekly(seasons=[2025])
print(rosters_weekly.columns)

with pl.Config(set_tbl_cols=-1):
    print(
        rosters_weekly.filter(pl.col("full_name").str.contains("Mahomes"))
        .sort("week")
    )

['season', 'team', 'position', 'depth_chart_position', 'jersey_number', 'status', 'full_name', 'first_name', 'last_name', 'birth_date', 'height', 'weight', 'college', 'gsis_id', 'espn_id', 'sportradar_id', 'yahoo_id', 'rotowire_id', 'pff_id', 'pfr_id', 'fantasy_data_id', 'sleeper_id', 'years_exp', 'headshot_url', 'ngs_position', 'week', 'game_type', 'status_description_abbr', 'football_name', 'esb_id', 'gsis_it_id', 'smart_id', 'entry_year', 'rookie_year', 'draft_club', 'draft_number']
shape: (17, 36)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ sea ┆ tea ┆ pos ┆ dep ┆ jer ┆ sta ┆ ful ┆ fir ┆ las ┆ bir ┆ hei ┆ wei ┆ col ┆ gsi ┆ esp ┆ spo ┆ yah ┆ rot ┆ pff ┆ pfr ┆ fan ┆ sle ┆ yea ┆ hea ┆ ngs ┆ wee ┆ gam ┆ sta ┆ foo ┆ esb ┆ gsi ┆ sma ┆ ent ┆ roo ┆ dra ┆ dra │
│ son ┆ m   ┆ iti ┆ th_ ┆ sey ┆ tus ┆ l_n ┆ st_ ┆ t_n ┆ t

In [86]:
import polars as pl
check = pl.read_csv("../data/player_features.csv")
with pl.Config(set_tbl_cols=-1):
    print(check.filter(pl.col("player_name").str.contains("Mahomes")))

shape: (1, 37)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pas ┆ pas ┆ pas ┆ rus ┆ rus ┆ rec ┆ tar ┆ rec ┆ rec ┆ fan ┆ pla ┆ pla ┆ pos ┆ gam ┆ tea ┆ is_ ┆ bas ┆ rou ┆ rou ┆ pos ┆ is_ ┆ pas ┆ rus ┆ qb_ ┆ coa ┆ con ┆ ret ┆ pos ┆ rec ┆ adp ┆ adp ┆ tim ┆ adp ┆ adp ┆ adp ┆ bye ┆ has │
│ sin ┆ sin ┆ sin ┆ hin ┆ hin ┆ ept ┆ get ┆ eiv ┆ eiv ┆ tas ┆ yer ┆ yer ┆ iti ┆ es_ ┆ m   ┆ roo ┆ eli ┆ nd  ┆ nd_ ┆ _ra ┆ sta ┆ s_a ┆ h_a ┆ cha ┆ ch_ ┆ tin ┆ urn ┆ iti ┆ ent ┆ --- ┆ _fo ┆ es_ ┆ _hi ┆ _lo ┆ _st ┆ --- ┆ _ad │
│ g_y ┆ g_t ┆ g_i ┆ g_y ┆ g_t ┆ ion ┆ s_p ┆ ing ┆ ing ┆ y_p ┆ _id ┆ _na ┆ on  ┆ pla ┆ --- ┆ kie ┆ ne_ ┆ --- ┆ buc ┆ nk  ┆ rte ┆ tt_ ┆ tt_ ┆ nge ┆ cha ┆ uit ┆ ing ┆ on_ ┆ _ma ┆ f64 ┆ rma ┆ dra ┆ gh  ┆ w   ┆ dev ┆ i64 ┆ p   │
│ ard ┆ ds_ ┆ nte ┆ ard ┆ ds_ ┆ s_p ┆ er_ ┆ _ya ┆ _td ┆ oin ┆ --- ┆ me  ┆ --- ┆ yed ┆ str

In [87]:
import polars as pl
from src.adp import PROJECT_ROOT

existing = pl.read_csv(PROJECT_ROOT / "data" / "player_features.csv")
dupes = existing.group_by("player_id").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
print(f"Duplicate player_id rows: {dupes.height}")
print(dupes)

if dupes.height > 0:
    sample_id = dupes["player_id"][0]
    print("\nExample duplicate:")
    print(existing.filter(pl.col("player_id") == sample_id))

Duplicate player_id rows: 0
shape: (0, 2)
┌───────────┬─────┐
│ player_id ┆ n   │
│ ---       ┆ --- │
│ str       ┆ u32 │
╞═══════════╪═════╡
└───────────┴─────┘


In [88]:
import polars as pl
import nflreadpy as nfl
from src.adp import PROJECT_ROOT
from src.rookies import CURRENT_ROOKIE_SEASON, OFFENSE_POSITIONS

# === 1. Confirm the duplicate rows are rookies ===
existing = pl.read_csv(PROJECT_ROOT / "data" / "player_features.csv")
dupe_ids = (
    existing.group_by("player_id").agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)["player_id"].to_list()
)
print("Are duplicate rows rookies?")
print(
    existing.filter(pl.col("player_id").is_in(dupe_ids))
    .select(["player_id", "player_name", "is_rookie"]).unique()
)

# === 2. Test the pfr_id join for null-key fan-out ===
season = CURRENT_ROOKIE_SEASON
players = nfl.load_players().filter(
    (pl.col("rookie_season") == season) &
    (pl.col("position").is_in(OFFENSE_POSITIONS))
).select([
    pl.col("gsis_id").alias("player_id"),
    pl.col("display_name").alias("player_name"),
    "position",
    "pfr_id",
])
print(f"\nTotal 2026 rookies (offense): {players.height}")
print(f"Rookies with null pfr_id: {players.filter(pl.col('pfr_id').is_null()).height}")

draft_picks = nfl.load_draft_picks().filter(pl.col("season") == season)
print(f"Draft picks for {season}: {draft_picks.height}, "
      f"with null pfr_player_id: {draft_picks.filter(pl.col('pfr_player_id').is_null()).height}")

draft_by_id = draft_picks.select([
    pl.col("pfr_player_id").alias("pfr_id"),
    pl.col("round").alias("round_by_id"),
])
joined = players.join(draft_by_id, on="pfr_id", how="left")
print(f"Row count after join on pfr_id: {joined.height} (started with {players.height}) "
      f"-- a jump here confirms null-key fan-out")

# === 3. Marvin Harrison Jr. / Travis Hunter ===
all_players = nfl.load_players()
print("\nMarvin Harrison entries:")
print(
    all_players.filter(pl.col("display_name").str.contains("Marvin Harrison"))
    .select(["gsis_id", "display_name", "position", "rookie_season", "last_season", "status"])
)

print("\nTravis Hunter entries:")
print(
    all_players.filter(pl.col("display_name").str.contains("Travis Hunter"))
    .select(["gsis_id", "display_name", "position", "rookie_season", "last_season", "status"])
)

Are duplicate rows rookies?
shape: (0, 3)
┌───────────┬─────────────┬───────────┐
│ player_id ┆ player_name ┆ is_rookie │
│ ---       ┆ ---         ┆ ---       │
│ str       ┆ str         ┆ bool      │
╞═══════════╪═════════════╪═══════════╡
└───────────┴─────────────┴───────────┘

Total 2026 rookies (offense): 224
Rookies with null pfr_id: 99
Draft picks for 2026: 257, with null pfr_player_id: 0
Row count after join on pfr_id: 224 (started with 224) -- a jump here confirms null-key fan-out

Marvin Harrison entries:
shape: (2, 6)
┌────────────┬─────────────────────┬──────────┬───────────────┬─────────────┬────────┐
│ gsis_id    ┆ display_name        ┆ position ┆ rookie_season ┆ last_season ┆ status │
│ ---        ┆ ---                 ┆ ---      ┆ ---           ┆ ---         ┆ ---    │
│ str        ┆ str                 ┆ str      ┆ i32           ┆ i32         ┆ str    │
╞════════════╪═════════════════════╪══════════╪═══════════════╪═════════════╪════════╡
│ 00-0007024 ┆ Marvin Harriso

In [89]:
import polars as pl
import nflreadpy as nfl
from src.rookies import CURRENT_ROOKIE_SEASON, OFFENSE_POSITIONS

season = CURRENT_ROOKIE_SEASON

players = nfl.load_players().filter(
    (pl.col("rookie_season") == season) & (pl.col("position").is_in(OFFENSE_POSITIONS))
).select([
    pl.col("gsis_id").alias("player_id"),
    pl.col("display_name").alias("player_name"),
    "position", pl.col("latest_team").alias("team"), "pfr_id",
])
print(f"Raw rookie rows: {players.height}, unique player_id: {players['player_id'].n_unique()}")

draft_picks = nfl.load_draft_picks().filter(pl.col("season") == season)
draft_by_name = draft_picks.select([
    pl.col("pfr_player_name").alias("player_name"), "position",
    pl.col("round").alias("round_by_name"),
])
dup_keys = (
    draft_by_name.group_by(["player_name","position"]).agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
)
print(f"\nDuplicate (player_name, position) keys in draft_by_name: {dup_keys.height}")
print(dup_keys)

draft_by_id = draft_picks.select([
    pl.col("pfr_player_id").alias("pfr_id"), pl.col("round").alias("round_by_id"),
])
step1 = players.join(draft_by_id, on="pfr_id", how="left")
step2 = step1.join(draft_by_name, on=["player_name","position"], how="left")
print(f"\nAfter join 1: {step1.height} rows. After join 2: {step2.height} rows.")

print("\nZachariah Branch (00-0041044) after join 2:")
print(step2.filter(pl.col("player_id") == "00-0041044"))

Raw rookie rows: 224, unique player_id: 224

Duplicate (player_name, position) keys in draft_by_name: 0
shape: (0, 3)
┌─────────────┬──────────┬─────┐
│ player_name ┆ position ┆ n   │
│ ---         ┆ ---      ┆ --- │
│ str         ┆ str      ┆ u32 │
╞═════════════╪══════════╪═════╡
└─────────────┴──────────┴─────┘

After join 1: 224 rows. After join 2: 224 rows.

Zachariah Branch (00-0041044) after join 2:
shape: (1, 7)
┌────────────┬──────────────────┬──────────┬──────┬──────────┬─────────────┬───────────────┐
│ player_id  ┆ player_name      ┆ position ┆ team ┆ pfr_id   ┆ round_by_id ┆ round_by_name │
│ ---        ┆ ---              ┆ ---      ┆ ---  ┆ ---      ┆ ---         ┆ ---           │
│ str        ┆ str              ┆ str      ┆ str  ┆ str      ┆ i32         ┆ i32           │
╞════════════╪══════════════════╪══════════╪══════╪══════════╪═════════════╪═══════════════╡
│ 00-0041044 ┆ Zachariah Branch ┆ WR       ┆ ATL  ┆ BranZa00 ┆ 3           ┆ 3             │
└────────────┴────

In [90]:
import polars as pl
import nflreadpy as nfl
from src.rookies import (
    get_current_rookie_class, assign_round_bucket, get_latest_depth_chart,
    CURRENT_ROOKIE_SEASON,
)

season = CURRENT_ROOKIE_SEASON
rookie_class = assign_round_bucket(get_current_rookie_class(season))

bad_ids = ["00-0041044","00-0041139","00-0041529","00-0041292","00-0041547",
           "00-0041104","00-0041228","00-0041397","00-0041512","00-0041051",
           "00-0040895","00-0041490","00-0040876","00-0041100","00-0041299",
           "00-0041040","00-0041099","00-0041185"]

# Which teams do the duplicated players belong to?
print("Team breakdown of the 18 duplicated players:")
print(rookie_class.filter(pl.col("player_id").is_in(bad_ids)).select(["player_id","player_name","team"]))

# Does the depth chart itself have duplicate player_id rows in the "latest" snapshot?
depth_charts = nfl.load_depth_charts(seasons=[season])
print(f"\nRaw depth_charts rows: {depth_charts.height}")
print("Columns:", depth_charts.columns)

latest_dt = depth_charts.group_by("team").agg(pl.col("dt").max().alias("latest_dt"))
latest = depth_charts.join(latest_dt, on="team").filter(pl.col("dt") == pl.col("latest_dt"))
print(f"\nLatest-snapshot depth chart rows: {latest.height}")

dupe_players_in_depth_chart = (
    latest.group_by("gsis_id").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
)
print(f"Players with multiple rows in the latest snapshot: {dupe_players_in_depth_chart.height}")

# Zoom in: what do Zachariah Branch's raw depth chart rows actually look like?
print("\nZachariah Branch's rows in the latest depth chart snapshot:")
print(latest.filter(pl.col("gsis_id") == "00-0041044"))

Team breakdown of the 18 duplicated players:
shape: (18, 3)
┌────────────┬────────────────────────┬──────┐
│ player_id  ┆ player_name            ┆ team │
│ ---        ┆ ---                    ┆ ---  │
│ str        ┆ str                    ┆ str  │
╞════════════╪════════════════════════╪══════╡
│ 00-0041185 ┆ Vinny Anthony II       ┆ ATL  │
│ 00-0041139 ┆ Damon Bankston         ┆ NYG  │
│ 00-0041044 ┆ Zachariah Branch       ┆ ATL  │
│ 00-0041099 ┆ Barion Brown           ┆ NO   │
│ 00-0041100 ┆ Josh Cameron           ┆ JAX  │
│ 00-0041104 ┆ Demond Claiborne       ┆ MIN  │
│ 00-0041529 ┆ Kevin Coleman Jr.      ┆ MIA  │
│ 00-0041547 ┆ KC Concepcion          ┆ CLE  │
│ 00-0041228 ┆ Sahmir Hagans          ┆ IND  │
│ 00-0041490 ┆ Eli Heidenreich        ┆ PIT  │
│ 00-0040895 ┆ Emmanuel Henderson Jr. ┆ SEA  │
│ 00-0041292 ┆ Kolbe Katsis           ┆ DEN  │
│ 00-0041512 ┆ Jadarian Price         ┆ SEA  │
│ 00-0041299 ┆ Cameron Ross           ┆ DEN  │
│ 00-0040876 ┆ Elijah Sarratt         ┆ BAL  │


In [91]:
import polars as pl
from src.adp import PROJECT_ROOT

existing = pl.read_csv(PROJECT_ROOT / "data" / "player_features.csv")
dupes = existing.group_by("player_id").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
print(f"Total rows: {existing.height}")
print(f"Duplicate player_id rows: {dupes.height}")

Total rows: 1083
Duplicate player_id rows: 0


In [92]:
import polars as pl
from src.adp import fetch_ffc_adp, build_gsis_lookup, PROJECT_ROOT

ffc_adp = fetch_ffc_adp()
gsis_lookup = build_gsis_lookup()
existing = pl.read_csv(PROJECT_ROOT / "data" / "player_features.csv")

# recreate the same match set attach_adp() produces, just to get player_ids
name_counts = gsis_lookup.group_by("name_key").agg(pl.len().alias("n"))
unique_names = name_counts.filter(pl.col("n") == 1).select("name_key")
matched = ffc_adp.join(unique_names, on="name_key", how="semi").join(
    gsis_lookup.select(["player_id", "name_key"]), on="name_key", how="inner"
)

missing = matched.join(existing.select("player_id"), on="player_id", how="anti")
print(missing.select(["player_id"]).join(
    gsis_lookup.select(["player_id", "display_name"]), on="player_id", how="left"
))

shape: (2, 2)
┌────────────┬───────────────┐
│ player_id  ┆ display_name  │
│ ---        ┆ ---           │
│ str        ┆ str           │
╞════════════╪═══════════════╡
│ 00-0040718 ┆ Travis Hunter │
│ 00-0040177 ┆ Jordan James  │
└────────────┴───────────────┘


In [93]:
import polars as pl
from src.adp import PROJECT_ROOT

df = pl.read_csv(PROJECT_ROOT / "data" / "player_features.csv")
print(f"Total rows: {df.height}, columns: {df.columns}")
print(f"Rows with ADP: {df.filter(pl.col('has_adp')).height}")
print("\nTop 10 by ADP:")
print(df.filter(pl.col("has_adp")).sort("adp").select(
    ["player_name", "position", "team", "adp", "fantasy_points_per_game"]
).head(10))

Total rows: 1083, columns: ['passing_yards_per_game', 'passing_tds_per_game', 'passing_interceptions_per_game', 'rushing_yards_per_game', 'rushing_tds_per_game', 'receptions_per_game', 'targets_per_game', 'receiving_yards_per_game', 'receiving_tds_per_game', 'fantasy_points_per_game', 'player_id', 'player_name', 'position', 'games_played', 'team', 'is_rookie', 'baseline_low_confidence', 'round', 'round_bucket', 'pos_rank', 'is_starter', 'pass_att_pg', 'rush_att_pg', 'qb_changed', 'coach_changed', 'continuity_score', 'returning_oline_starters', 'position_competition_ppg', 'recent_major_injury', 'adp', 'adp_formatted', 'times_drafted', 'adp_high', 'adp_low', 'adp_stdev', 'bye', 'has_adp']
Rows with ADP: 190

Top 10 by ADP:
shape: (10, 5)
┌────────────────┬──────────┬──────┬──────┬─────────────────────────┐
│ player_name    ┆ position ┆ team ┆ adp  ┆ fantasy_points_per_game │
│ ---            ┆ ---      ┆ ---  ┆ ---  ┆ ---                     │
│ str            ┆ str      ┆ str  ┆ f64  ┆ 

In [110]:
import polars as pl
data = pl.read_csv("../data/backtest_features.csv")
data = data.filter(pl.col("actual_games_played") >= 8)
print(data.group_by("position").len().sort("position"))

shape: (4, 2)
┌──────────┬─────┐
│ position ┆ len │
│ ---      ┆ --- │
│ str      ┆ u32 │
╞══════════╪═════╡
│ QB       ┆ 96  │
│ RB       ┆ 239 │
│ TE       ┆ 216 │
│ WR       ┆ 396 │
└──────────┴─────┘


In [114]:
import statsmodels.api as sm
import polars as pl

data = pl.read_csv("../data/backtest_features.csv")

PREDICTORS = [
    "pass_att_pg", "rush_att_pg", "qb_changed", "coach_changed",
    "returning_oline_starters", "position_competition_ppg",
]

def run_position_regression(data, position):
    subset = data.filter(pl.col("position") == position)
    df = subset.select(PREDICTORS + ["delta"]).to_pandas()

    # statsmodels needs numeric, not bool
    df["qb_changed"] = df["qb_changed"].astype(int)
    df["coach_changed"] = df["coach_changed"].astype(int)

    X = sm.add_constant(df[PREDICTORS])
    y = df["delta"]

    model = sm.OLS(y, X).fit()
    print(f"===== {position} (n={len(df)}) =====")
    print(model.summary())
    print()
    return model

models = {}
for pos in ["QB", "RB", "WR", "TE"]:
    models[pos] = run_position_regression(data, pos)

===== QB (n=202) =====
                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.028
Method:                 Least Squares   F-statistic:                    0.1025
Date:                Thu, 30 Jul 2026   Prob (F-statistic):              0.996
Time:                        15:13:53   Log-Likelihood:                -675.27
No. Observations:                 202   AIC:                             1365.
Df Residuals:                     195   BIC:                             1388.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
c

In [112]:
data_filtered = pl.read_csv("../data/backtest_features.csv").filter(pl.col("actual_games_played") >= 8)
print("Rows after filter:", data_filtered.shape[0])
print(data_filtered.group_by("position").len().sort("position"))

qb_model = run_position_regression(data_filtered, "QB")

Rows after filter: 947
shape: (4, 2)
┌──────────┬─────┐
│ position ┆ len │
│ ---      ┆ --- │
│ str      ┆ u32 │
╞══════════╪═════╡
│ QB       ┆ 96  │
│ RB       ┆ 239 │
│ TE       ┆ 216 │
│ WR       ┆ 396 │
└──────────┴─────┘
===== QB (n=96) =====
                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                 -0.034
Method:                 Least Squares   F-statistic:                    0.4814
Date:                Thu, 30 Jul 2026   Prob (F-statistic):              0.821
Time:                        15:12:36   Log-Likelihood:                -288.03
No. Observations:                  96   AIC:                             590.1
Df Residuals:                      89   BIC:                             608.0
Df Model:                           6                                         
Covariance Type:            nonrobust   

In [113]:
print(data_filtered.select("actual_games_played").describe())

shape: (9, 2)
┌────────────┬─────────────────────┐
│ statistic  ┆ actual_games_played │
│ ---        ┆ ---                 │
│ str        ┆ f64                 │
╞════════════╪═════════════════════╡
│ count      ┆ 947.0               │
│ null_count ┆ 0.0                 │
│ mean       ┆ 13.611404           │
│ std        ┆ 2.906397            │
│ min        ┆ 8.0                 │
│ 25%        ┆ 11.0                │
│ 50%        ┆ 14.0                │
│ 75%        ┆ 16.0                │
│ max        ┆ 18.0                │
└────────────┴─────────────────────┘


In [115]:
rb_model = run_position_regression(data_filtered, "RB")
wr_model = run_position_regression(data_filtered, "WR")
te_model = run_position_regression(data_filtered, "TE")

===== RB (n=239) =====
                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     1.442
Date:                Thu, 30 Jul 2026   Prob (F-statistic):              0.199
Time:                        15:15:27   Log-Likelihood:                -664.10
No. Observations:                 239   AIC:                             1342.
Df Residuals:                     232   BIC:                             1367.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
c

In [116]:
REDUCED_PREDICTORS = ["qb_changed", "coach_changed", "returning_oline_starters"]

def run_reduced_regression(data, position):
    subset = data.filter(pl.col("position") == position)
    df = subset.select(REDUCED_PREDICTORS + ["delta"]).to_pandas()
    df["qb_changed"] = df["qb_changed"].astype(int)
    df["coach_changed"] = df["coach_changed"].astype(int)

    X = sm.add_constant(df[REDUCED_PREDICTORS])
    y = df["delta"]

    model = sm.OLS(y, X).fit()
    print(f"===== {position} reduced (n={len(df)}) =====")
    print(model.summary())
    print()
    return model

reduced_models = {}
for pos in ["QB", "RB", "WR", "TE"]:
    reduced_models[pos] = run_reduced_regression(data_filtered, pos)

===== QB reduced (n=96) =====
                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.022
Method:                 Least Squares   F-statistic:                    0.3057
Date:                Thu, 30 Jul 2026   Prob (F-statistic):              0.821
Time:                        15:21:05   Log-Likelihood:                -289.08
No. Observations:                  96   AIC:                             586.2
Df Residuals:                      92   BIC:                             596.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [117]:
rb_subset = data_filtered.filter(pl.col("position") == "RB").to_pandas()
rb_subset["qb_changed"] = rb_subset["qb_changed"].astype(int)
rb_subset["coach_changed"] = rb_subset["coach_changed"].astype(int)

print(rb_subset[["qb_changed", "coach_changed"]].corr())

rb_subset["continuity_disruption"] = rb_subset["qb_changed"] + rb_subset["coach_changed"]

X = sm.add_constant(rb_subset[["continuity_disruption"]])
y = rb_subset["delta"]
combined_model = sm.OLS(y, X).fit()
print(combined_model.summary())

               qb_changed  coach_changed
qb_changed       1.000000       0.060476
coach_changed    0.060476       1.000000
                            OLS Regression Results                            
Dep. Variable:                  delta   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     7.950
Date:                Thu, 30 Jul 2026   Prob (F-statistic):            0.00522
Time:                        15:24:24   Log-Likelihood:                -664.53
No. Observations:                 239   AIC:                             1333.
Df Residuals:                     237   BIC:                             1340.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.97

In [1]:
import requests
def pull(t):
    r = requests.get("https://fantasyfootballcalculator.com/api/v1/adp/ppr",
                     params={"teams": t, "year": 2026, "position": "all"}, timeout=15)
    return {p["name"]: p["adp"] for p in r.json()["players"]}
a, b = pull(8), pull(12)
same = sum(1 for n in a if n in b and a[n] == b[n])
print(f"{same} of {len(a)} players have byte-identical ADP across the two feeds")

256 of 256 players have byte-identical ADP across the two feeds
